# Multimodal Model – Combining Time-Series, Static Variables, Images, and Text

In this lab, we explore a multimodal model that integrates time-series data, static variables, images, and text to predict mortgage loan delinquency.

This is a completed lab from the DataCamp webinar *Multimodal Deep Learning for Credit Scoring*, itself adapted from the textbook *Deep Learning in Banking: Integrating Artificial Intelligence for Next Generation Financial Services* by Cristian Bravo, Sebastian Maldonado, and María Óskarsdóttir (www.bankingbook.ml). See the [README](../README.md) for full attribution, results, and known limitations.

**This notebook has been refactored** from the original single-notebook lab into a proper `src/` package (data pipeline, dataset, models, training, evaluation) -- this notebook is now a thin walkthrough that imports and calls that package, rather than redefining everything inline. It is provided **unexecuted**: the underlying data isn't bundled in this repo (see `src/data/download.py`), so run it end-to-end after installing `requirements.txt` and downloading the data. The results reported in the README reflect the original lab's run.

We start by installing and importing the necessary libraries.

In [ ]:
!pip install -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image
%matplotlib inline

from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve, classification_report
from transformers import AutoTokenizer

from src.config import TIME_SERIES_COLS

In [ ]:
# Folder to save downloaded data and model outputs (checkpoints, plots).
DATA_DIR = "../data"
bookPath = "../outputs"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(bookPath, exist_ok=True)

We will start by loading and preprocessing the four data sources:

*   Time Series Data: Loan performance history over time with Transformer Encoder.
*   Static Features: Borrower information, property details, and loan
*   LiDAR Images: LiDAR images processed using a CNN.
*   Textual Data: Federal Reserve speeches analyzed with DistilBERT.

## Cleaning Time Series data

Let's start by downloading the time-series dataset from Freddie Mac. This dataset provides monthly loan status updates covering the period from November 2021 to June 2024.

In [ ]:
from src.data.download import download_time_series_csv

time_series_csv_path = download_time_series_csv(data_dir=DATA_DIR)

In [ ]:
from src.data.time_series import load_loan_performance

df = load_loan_performance(time_series_csv_path)
df

We assign a value of 1 to loans that were delinquent at least once in the last three months (April to June 2024). Our goal is to predict whether a loan will become delinquent in the next quarter.

In [ ]:
from src.data.time_series import label_delinquency

df = label_delinquency(df)
df

We normalize the data and one-hot encode the labels using scikit-learn's `OneHotEncoder`. We will also set the class weights, since delinquency is a rare event.

In [ ]:
from src.data.time_series import build_loan_sequences

x_df = build_loan_sequences(df, num_cols=TIME_SERIES_COLS)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

from src.data.time_series import scale_dataframe_sequences

x_train_val, x_test = scale_dataframe_sequences(
    x_df[TIME_SERIES_COLS], num_cols=TIME_SERIES_COLS, test_var=x_df["if_test"]
)

enc = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
y_train_val = enc.fit_transform(x_df.loc[x_df["if_test"] == 0, "target"].values.reshape(-1, 1))[:, 1]
y_test = enc.transform(x_df.loc[x_df["if_test"] == 1, "target"].values.reshape(-1, 1))[:, 1]

pos_weight = torch.tensor(np.sum(1 - y_train_val) / np.sum(y_train_val), dtype=torch.float32)

In [ ]:
x_train_val

## Cleaning static variables

Next, we will clean static features such as credit score, property type, and occupancy status, which were recorded at the time of the loan application.

In [ ]:
from src.data.download import download_static_csv

static_csv_path = download_static_csv(data_dir=DATA_DIR)

In [ ]:
from src.data.static_features import load_static_features

static_df = load_static_features(static_csv_path)
static_df

We normalize numeric variables and one-hot encode categorical variables to ensure consistency in the dataset.

In [ ]:
from src.config import STATIC_CATEGORICAL_COLS, STATIC_NUMERIC_COLS

STATIC_NUMERIC_COLS, STATIC_CATEGORICAL_COLS

In [ ]:
# Add LOAN_NUMBER back before merging
x_df["LOAN_NUMBER"] = x_df.index

loan_numbers_train = x_df.loc[x_df["if_test"] == 0, "LOAN_NUMBER"]
loan_numbers_test = x_df.loc[x_df["if_test"] == 1, "LOAN_NUMBER"]

from src.data.static_features import prepare_static_features

static_train_final, static_test_final = prepare_static_features(
    static_df, loan_numbers_train, loan_numbers_test
)

We then merge the cleaned static variables with the time-series table to create a unified dataset for modeling.

In [ ]:
print(f"Processed Train Static Data Shape: {static_train_final.shape}")
print(f"Processed Test Static Data Shape: {static_test_final.shape}")

In [ ]:
from src.data.static_features import merge_time_series_with_static

x_train_val, x_test = merge_time_series_with_static(
    x_train_val, x_test, loan_numbers_train, loan_numbers_test, static_train_final, static_test_final
)

print(f"x_train Shape After Merge: {x_train_val.shape}")
print(f"x_test Shape After Merge: {x_test.shape}")

## Merging image dataset

Next, we will merge the image mapping dataset using MSA (Metropolitan Statistical Area) and ZIP3 (the first three digits of the ZIP code) to align LiDAR images with the corresponding loan records. Note this links images at the metro-area/ZIP3 level, not per individual loan -- see the README "Limitations" section.

In [ ]:
from src.data.download import download_lidar_images, download_image_mapping_csv

lidar_dir = download_lidar_images(data_dir=DATA_DIR)
image_mapping_csv_path = download_image_mapping_csv(data_dir=DATA_DIR)

In [ ]:
from src.data.images import load_image_mapping

image_df = load_image_mapping(image_mapping_csv_path)
image_df

**TASK: Display the first image.**

Note: `LiDAR_File` paths in the mapping CSV are relative to the directory the original `usgs_lidar.zip` was extracted into. If the image below doesn't load, adjust `DATA_DIR` (or your working directory) to match where `download_lidar_images` extracted the archive.

In [ ]:
Image(filename=image_df.iloc[0]["LiDAR_File"])

In [ ]:
from src.data.images import merge_images

x_train_val, x_test = merge_images(x_train_val, x_test, image_df)

Finally, we have established a comprehensive dataset that integrates time-series data, static variables, and image mapping, providing a unified foundation for our multimodal analysis.

In [ ]:
x_train_val

## Get fixed textual data

We will use the most recent Federal Reserve speech from March 2024 as the fixed textual data for our analysis.

**Scenario**:

You are a data scientist at a bank, analyzing loan risk. After watching the Federal Reserve speech from March 2024, you aim to leverage multiple datasets -- including time-series data from the loan dataset, static features from loan applications, LiDAR images, and the Fed speech -- to predict whether a loan will become delinquent in the next quarter.

In [ ]:
from src.data.download import download_fed_speeches_csv

fed_speeches_csv_path = download_fed_speeches_csv(data_dir=DATA_DIR)

In [ ]:
from src.data.text import load_fed_speeches

fed_speech = load_fed_speeches(fed_speeches_csv_path)
fed_speech

In [ ]:
fed_speech[(fed_speech.year == 2024) & (fed_speech.month == 3)]

In [ ]:
from src.data.text import select_speech

# Most recent speech matching year/month
fixed_fed_speech = select_speech(fed_speech, year=2024, month=3)

In this step, we preprocess the speech text by **tokenizing, removing stopwords, and eliminating punctuation** to prepare the data for further analysis.

In [ ]:
from src.data.text import clean_text

fixed_fed_speech = clean_text(fixed_fed_speech)
fixed_fed_speech

## Preparing Dataloader for multimodal model

We create a **`MultimodalDataset`** class (in `src/dataset.py`), integrating time-series data, static features, LiDAR images, and textual data.

In [ ]:
from src.dataset import build_fixed_text_tokens

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
fixed_tokens = build_fixed_text_tokens(fixed_fed_speech, tokenizer=tokenizer)

This splits the dataset, creates PyTorch dataset instances, and prepares data loaders for training, validation, and testing in a multimodal deep learning model.

In [ ]:
from src.train import build_dataloaders, set_seed

SEED = 42
set_seed(SEED)

train_loader, val_loader, test_loader = build_dataloaders(
    x_train_val, x_test, y_train_val, y_test, fixed_tokens, seed=SEED
)

# Check batch structure
batch = next(iter(train_loader))
print(f"Time-series data shape: {batch['time_series'].shape}")
print(f"Static feature shape: {batch['static_features'].shape}")
print(f"Text input IDs shape: {batch['text_data']['input_ids'].shape}")
print(f"Image data shape: {batch['image_data'].shape}")
print(f"Target shape: {batch['target'].shape}")

## Multimodal deep learning model - intermediate fusion

In this section, we build a multimodal deep learning model that integrates four distinct data modalities.

To predict whether a loan will become delinquent in the following quarter, we combine Transformers for time-series data, CNNs for image processing, and a fusion layer to integrate information from all modalities.

This is our architecture:

![Concat fusion architecture](../docs/images/architecture_concat_fusion.png)

In [ ]:
from src.models.fusion_concat import MultimodalDelinquencyModel

Now, let's train the model! Given its complexity, training may take some time.

To improve efficiency, `src.train.train_model` uses **mixed precision training** on CUDA (`torch.amp.GradScaler`), which accelerates computation while reducing memory consumption.

In [ ]:
from src.models import build_model
from src.train import train_model

num_time_series_features = len(TIME_SERIES_COLS)
seq_length = len(x_train_val.iloc[0, 0])
num_static_features = len(x_train_val.iloc[0, 8:-1])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model("concat", num_time_series_features, seq_length, num_static_features, dropout=0.3).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

checkpoint_path = os.path.join(bookPath, "best_multimodal_model.pth")

In [ ]:
### One epoch takes 1.5 min with A100
# train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=1, checkpoint_path=checkpoint_path)

Training for 10 epochs takes approximately **15 minutes**. However, we observe that further training may be beneficial, as the validation loss continues to decrease.

Now, let's proceed with evaluating the model. You can load a fully-trained checkpoint instead of training from scratch (which takes around two hours) -- this is always good practice for reproducing published results.

In [ ]:
from src.evaluate import evaluate_model

In [ ]:
from src.data.download import download_pretrained_checkpoint

checkpoint_path = download_pretrained_checkpoint("concat", output_dir=bookPath)
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
print("Loaded best model for evaluation!")

all_labels, all_preds, all_probs = evaluate_model(model, test_loader, device)

In [ ]:
### Confusion Matrix (Normalized)
cm = confusion_matrix(all_labels, all_preds)
cm_rates = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_rates, annot=True, fmt=".2f", cmap=sns.light_palette("seagreen", as_cmap=True),
            xticklabels=["No Delinquency", "Delinquent"],
            yticklabels=["No Delinquency", "Delinquent"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Normalized Confusion Matrix (Rates)")
plt.savefig(os.path.join(bookPath, "multimodal_confusion_matrix.pdf"))
plt.show()

In [ ]:
### ROC Curve & AUC Score
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score = roc_auc_score(all_labels, all_probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve - Delinquency Prediction")
plt.legend(loc=4)
plt.grid()
plt.savefig(os.path.join(bookPath, "multimodal_roc_curves.pdf"))
plt.show()

### Classification Report
print("Classification Report:\n", classification_report(all_labels, all_preds, digits=4))

Our model performs pretty well! (See the README results table for the numbers from the original run.)

## Multimodal model with cross-attention

In this section, we implement a multimodal deep learning model that integrates information from multiple data sources using cross-attention mechanisms. Unlike traditional models that process different modalities independently, cross-attention lets the model dynamically learn relationships between feature representations, improving predictive performance.

### Model Architecture

Our model processes four distinct modalities:

*  Time-Series Data: Loan performance over time, modeled using a Transformer with positional encoding.
*  Static Features: Borrower and loan characteristics, processed through a Multi-Layer Perceptron (MLP).
*  Text Data: Federal Reserve speeches, encoded using DistilBERT.
*  LiDAR Images: Spatial features extracted via a CNN.

To enhance feature interactions, cross-attention is applied at multiple levels:

*  Text ↔ Time-Series: Captures relationships between Federal Reserve statements and historical loan performance.
*  Static Features ↔ LiDAR Images: Links borrower characteristics with geographic-based insights.
*  Time-Series ↔ Static Features: Models how borrower attributes influence loan behavior over time.

### Fusion Strategy

After applying cross-attention, the model undergoes two levels of feature refinement:

*  Second-Level Cross-Attention: Further interactions between attended representations from different modalities.
*  Weighted Fusion Mechanism: A learnable attention-based fusion layer assigns adaptive importance to each attended feature representation before making a final prediction.
*  The fused representation is then passed through a fully connected prediction layer to estimate loan delinquency status.

This is the architecture:

![Cross-attention fusion architecture](../docs/images/architecture_cross_attention.png)

In [ ]:
from src.models.fusion_cross_attention import CrossAttentionFusionModel

In [ ]:
import gc

del model
del optimizer
del train_loader

gc.collect()
torch.cuda.empty_cache()

In [ ]:
SEED = 42
set_seed(SEED)

train_loader, val_loader, test_loader = build_dataloaders(
    x_train_val, x_test, y_train_val, y_test, fixed_tokens, seed=SEED
)

model = build_model("cross_attention", num_time_series_features, seq_length, num_static_features, dropout=0.3).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5)

checkpoint_path = os.path.join(bookPath, "best_multimodal_cross_attention_model.pth")

In [ ]:
# Train - takes 1.5 min per epoch with A100
# train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=1, checkpoint_path=checkpoint_path)

Like before, if you want to download a fully-trained model, leave the first line uncommented.

In [ ]:
checkpoint_path = download_pretrained_checkpoint("cross_attention", output_dir=bookPath)
model.load_state_dict(torch.load(checkpoint_path, weights_only=True))
print("Loaded best model for evaluation!")

all_labels, all_preds, all_probs = evaluate_model(model, test_loader, device)

In [ ]:
### Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
cm_rates = cm.astype("float") / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_rates, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=["No Delinquency", "Delinquent"],
            yticklabels=["No Delinquency", "Delinquent"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Normalized Confusion Matrix (Rates)")
plt.savefig(os.path.join(bookPath, "multimodal_confusion_matrix_crossattention.pdf"))
plt.show()

In [ ]:
### ROC Curve & AUC Score
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score = roc_auc_score(all_labels, all_probs)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {auc_score:.3f})", linewidth=2)
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate (FPR)")
plt.ylabel("True Positive Rate (TPR)")
plt.title("ROC Curve - Delinquency Prediction")
plt.legend(loc=4)
plt.grid()
plt.savefig(os.path.join(bookPath, "multimodal_roc_curve_crossattention.pdf"))
plt.show()

### Classification Report
print("Classification Report:\n", classification_report(all_labels, all_preds, digits=4))

We observe an improvement in the ROC AUC score with cross-attention fusion. Now you know how to run multimodal models!

See the [README](../README.md) for the results table, known limitations, and the roadmap for extending this into an original contribution.